# 나이브 베이즈 알고리즘

In [2]:
# 실습에서 사용한 코드
# import numpy as np
# from sklearn.datasets import load_breast_cancer
# from sklearn.model_selection import train_test_split
# from sklearn.naive_bayes import GaussianNB
# from sklearn.metrics import accuracy_score, confusion_matrix
# from sklearn.model_selection import StratifiedKFold
# from sklearn.model_selection import StratifiedShuffleSplit

In [21]:
# 실습 1 Breast Cancer 활용

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()


print(bc.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [22]:
# 실습 1-1 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(bc.data,
                                                    bc.target,
                                                    stratify=bc.target,
                                                    random_state=430)

print(x_train.shape, y_test.shape)

(426, 30) (143,)


In [23]:
# 실습 1-2 나이브 베이즈 적용

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

bc_nb = GaussianNB()           # 연속된 수치 데이터들을 정규 확률 분포로 변경하여 전처리해줌
bc_nb.fit(x_train, y_train)
pred = bc_nb.predict(x_test)

print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

0.9230769230769231
[[46  7]
 [ 4 86]]


In [28]:
# 실습 1-3 Stratified-K-fold 적용

from sklearn.model_selection import StratifiedShuffleSplit # 인덱스를 섞어줌

acc = []
split = StratifiedShuffleSplit(10)

for train_idx, test_idx in split.split(bc.data, bc.target):
  bc_nb = GaussianNB()           # 연속된 수치 데이터들을 정규 확률 분포로 변경하여 전처리해줌
  bc_nb.fit(bc.data[train_idx], bc.target[train_idx])
  pred = bc_nb.predict(bc.data[test_idx])
  print(accuracy_score(bc.target[test_idx], pred))
  acc.append(accuracy_score(bc.target[test_idx], pred))

print(np.mean(acc))

0.9473684210526315
0.9649122807017544
0.9473684210526315
0.9473684210526315
1.0
1.0
0.9473684210526315
0.9473684210526315
0.9649122807017544
0.9649122807017544
0.9631578947368421


In [45]:
# 실습 1-3 Stratified-K-fold 적용(나의 풀이)

import numpy as np
from sklearn.model_selection import StratifiedKFold

bc = load_breast_cancer()
X = bc.data
y = bc.target

# 2. 모델 및 K-Fold 객체 생성
bc_nb = GaussianNB()
skf = StratifiedShuffleSplit(10)
cv_accuracy = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y), 1):

    # 인덱스를 기반으로 학습용/검증용 데이터 분할
    x_train, x_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # 모델 학습 및 예측
    bc_nb.fit(x_train, y_train)
    pred = bc_nb.predict(x_test)

    # 정확도 계산 및 리스트에 추가
    acc = accuracy_score(y_test, pred)
    cv_accuracy.append(acc)
    print(f"{fold_idx}번 : {acc:.4f}")  # 소수점 4자리까지

# 4. 최종 평균 정확도 산출
print("\n 평균 정확도:", round(np.mean(cv_accuracy), 4)) # 소수점 4자리까지

1번 : 0.8947
2번 : 0.9649
3번 : 0.9123
4번 : 0.9123
5번 : 0.9123
6번 : 0.9649
7번 : 0.9123
8번 : 0.9474
9번 : 0.9474
10번 : 0.9123

 평균 정확도: 0.9281


In [49]:
# 실습 2-1 KNN 한번에 해결하는 함수

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'n_neighbors':range(1,50,2),}              # KNN 중요한 파라미터를 입력해줌 KNeighborsClassifier() 내부 값을 정해줌
]

scaler = StandardScaler()
bc_data_std = scaler.fit_transform(bc.data)     # 데이터 표준화


bc.knn = KNeighborsClassifier()
grid_m = GridSearchCV(bc.knn,                   # 적용할 모델 KNeighborsClassifier()
                      param_grid = param_grid,  # 위에서 정한 파라미터 설정
                      cv=10,                    # 10번 교차검증
                      return_train_score=True,  # 훈련 점수도 표기
                      verbose=3)                # 작업하는 거 보게 하는 설정

grid_m.fit(bc_data_std, bc.target)              # 실행

Fitting 10 folds for each of 50 candidates, totalling 500 fits
[CV 1/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.965) total time=   0.0s
[CV 2/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.930) total time=   0.0s
[CV 3/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 4/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.982) total time=   0.0s
[CV 5/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=1.000) total time=   0.0s
[CV 6/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 7/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.912) total time=   0.0s
[CV 8/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 9/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 10/10] END metric=minkowski, n_neighbors=1

GridSearchCV(cv=10, estimator=KNeighborsClassifier(),
             param_grid=[{'metric': ['minkowski', 'chebyshev'],
                          'n_neighbors': range(1, 50, 2)}],
             return_train_score=True, verbose=3)

In [48]:
# 실습 2-1 가장 좋은 K 과 평균 정확도
grid_m.best_params_, grid_m.best_score_

({'n_neighbors': 7}, np.float64(0.9683897243107771))

In [51]:
# 실습 2-2 거리계산 방식 + 셔플 방식 추가

p_grid = [
    {'n_neighbors':range(1,50,2),
     'metric':['minkowski', 'chebyshev']}
]

split = StratifiedShuffleSplit(10)                # 셔플 적용
scaler = StandardScaler()
bc_data_std = scaler.fit_transform(bc.data)

bc.knn = KNeighborsClassifier()
grid_m = GridSearchCV(bc.knn,
                      param_grid = p_grid,
                      cv=split,                    # 셔플로 돌리기
                      return_train_score=True,
                      verbose=3)

grid_m.fit(bc_data_std, bc.target)

Fitting 10 folds for each of 50 candidates, totalling 500 fits
[CV 1/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 2/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.912) total time=   0.0s
[CV 3/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.930) total time=   0.0s
[CV 4/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.912) total time=   0.0s
[CV 5/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.895) total time=   0.0s
[CV 6/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.930) total time=   0.0s
[CV 7/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.947) total time=   0.0s
[CV 8/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.930) total time=   0.0s
[CV 9/10] END metric=minkowski, n_neighbors=1;, score=(train=1.000, test=0.930) total time=   0.0s
[CV 10/10] END metric=minkowski, n_neighbors=1

GridSearchCV(cv=StratifiedShuffleSplit(n_splits=10, random_state=None, test_size=None,
            train_size=None),
             estimator=KNeighborsClassifier(),
             param_grid=[{'metric': ['minkowski', 'chebyshev'],
                          'n_neighbors': range(1, 50, 2)}],
             return_train_score=True, verbose=3)

In [52]:
# 실습 2-2 결과
grid_m.best_params_, grid_m.best_score_

({'metric': 'minkowski', 'n_neighbors': 5}, np.float64(0.9614035087719299))

In [ ]:
# GridSearchCV(

#   estimator: Pipeline,                    = 테스트할 모델

#   param_grid: list[dict[str, Unknown]],   = 파라미터 후보 목록을 전달하는 곳

#   *,

#   scoring: Unknown | None = None,         = 채점 기준

#   n_jobs: Unknown | None = None,          = 연산 속도

#   refit: bool = True,                     = 재학습 여부

#   cv: Unknown | None = None,              = 교차검증

#   verbose: int = 0,                       = 컴퓨터가 일하는 과정 중계 단계 조정

#   pre_dispatch: str = "2*n_jobs",         = 일감을 적절히 나누어 대기

#   error_score: float = np.nan,            = 에러가 나면 어떻게 할지 대책

#   return_train_score: bool = False        = 훈련 점수도 성적표에 같이 적어줄지 여부
# )

In [56]:
# 실습 3-1 파이프라인

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# 파이프라인 구조 설정(기본값 설정)

pip = Pipeline([('preprocess', StandardScaler()),         # 전처리 방법 적용
                ('classifier', KNeighborsClassifier())    # 파이프라인 모델 적용
                ])

g_pram = [
    {"preprocess":[StandardScaler(),MinMaxScaler()],      # 전처리는 2가지 방법 적용
     "classifier":[KNeighborsClassifier()],
     "classifier__n_neighbors": range(1,50,2)},
    {"preprocess":[None],
     "classifier":[GaussianNB()]}
]

grid_m = GridSearchCV(pip,                              # 모델 설정
                      param_grid = g_pram,
                      cv=split,
                      return_train_score=True,
                      verbose=3)

grid_m.fit(bc.data, bc.target)
print(grid_m.best_params_, grid_m.best_score_)

Fitting 10 folds for each of 51 candidates, totalling 510 fits
[CV 1/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=1.000) total time=   0.0s
[CV 2/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.982) total time=   0.0s
[CV 3/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.982) total time=   0.0s
[CV 4/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.965) total time=   0.0s
[CV 5/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.982) total time=   0.0s
[CV 6/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.895) total time=   

# 결정트리

In [62]:
# 실습 4 결정트리

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()                               # max_depth 로 overfitting 방지가능
x_train, x_test, y_train, y_test = train_test_split(bc.data,
                                                    bc.target,
                                                    stratify=bc.target,
                                                    random_state=430)
dt.fit(x_train, y_train)
pred = dt.predict(x_test)

print(accuracy_score(y_test, pred))

0.951048951048951


In [69]:
# 실습 4-1 결정트리와 파이프라인

pip = Pipeline([('preprocess', StandardScaler()),
                ('classifier', KNeighborsClassifier())
                ])

g_pram = [
    {"preprocess":[StandardScaler()],           # 전처리
     "classifier":[KNeighborsClassifier()],     # 모델
     "classifier__n_neighbors": range(1,30,2)}, # 모델 설정

    {"preprocess":[None],                       # 전처리 필요없음
     "classifier":[GaussianNB()]},              # 모델

    {"preprocess":[None],                       # 전처리 필요없음
     "classifier" :[DecisionTreeClassifier()],  # 모델
     "classifier__max_depth": range(1,20)}      # 모델 설정
]

grid_m = GridSearchCV(pip,                      # 구조를 반영
                      param_grid = g_pram,      # grid 설정 적용
                      cv=split,                 # StratifiedKFold 방식 적용
                      return_train_score=True,  # 훈련점수도 기록
                      verbose=3)                # 과정의 디테일 출력

grid_m.fit(bc.data, bc.target)
print(grid_m.best_params_, grid_m.best_score_)

Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 1/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.912) total time=   0.0s
[CV 2/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.982) total time=   0.0s
[CV 3/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.947) total time=   0.0s
[CV 4/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.982) total time=   0.0s
[CV 5/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.947) total time=   0.0s
[CV 6/10] END classifier=KNeighborsClassifier(), classifier__n_neighbors=1, preprocess=StandardScaler();, score=(train=1.000, test=0.947) total time=   